# Setup / packages

In [1]:
%pip install pandas
import pandas as pd



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Load data target dataset (healthy life expectancy)

In [2]:
# Load tsv.gz file
df = pd.read_csv("eurostat.tsv.gz",sep="\t",compression="gzip")

df.head(10)

,"freq,unit,sex,indic_he,geo\TIME_PERIOD",2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,"A,YR,F,HLY_0,AT",57.8,58.1,57.1,56.8,57.0,58.0,59.3,61.3,61.3,60.5
1,"A,YR,F,HLY_0,BE",63.7,64.0,63.7,64.0,63.7,62.8 b,64.0,64.4,63.3,63.5
2,"A,YR,F,HLY_0,BG",66.1,65.0,67.5 b,66.2,67.6,68.4,67.8,65.1,68.9 b,71.0
3,"A,YR,F,HLY_0,CH",56.7,57.8,57.7,59.4,59.5,60.0,60.7,59.3,56.7,55.4
4,"A,YR,F,HLY_0,CY",66.1,63.4,68.8,65.8,62.4,63.0,63.1,66.8,66.3,65.7
5,"A,YR,F,HLY_0,CZ",65.0,63.7,64.0,62.4,63.4,62.6,62.5,63.4,62.4,62.6
6,"A,YR,F,HLY_0,DE",56.5,67.5 b,67.3,66.7,66.3,67.1,66.8 b,66.5,61.2 bu,63.0 b
7,"A,YR,F,HLY_0,DK",61.4,57.6,60.3,59.7,59.1,58.8,57.7 b,54.8,54.6,55.4
8,"A,YR,F,HLY_0,EE",57.1,56.2,59.0,57.2,55.0,57.7,59.6,58,60.6,59.6
9,"A,YR,F,HLY_0,EL",64.9,64.1,64.7,65.1,65.9,66.4,66.8,66.6,67.8,67.3


### Clean target data

In [3]:
#name of first column
df.columns[0]

'freq,unit,sex,indic_he,geo\\TIME_PERIOD'

In [4]:
# Remove time period and splot the columns by the comma
meta_name = df.columns[0]
meta_names = meta_name.replace("\\TIME_PERIOD", "").split(",")

# split value in first column
meta_cols = df.iloc[:, 0].str.split(",", expand=True)
meta_cols.columns = meta_names

# Remove original packed metadata column
df_clean = df.drop(df.columns[0], axis=1)

#combine the split columns wit the remaining
df_clean = pd.concat([meta_cols, df_clean], axis=1)

df_clean.head()

,freq,unit,sex,indic_he,geo,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,A,YR,F,HLY_0,AT,57.8,58.1,57.1,56.8,57.0,58.0,59.3,61.3,61.3,60.5
1,A,YR,F,HLY_0,BE,63.7,64.0,63.7,64.0,63.7,62.8 b,64.0,64.4,63.3,63.5
2,A,YR,F,HLY_0,BG,66.1,65.0,67.5 b,66.2,67.6,68.4,67.8,65.1,68.9 b,71.0
3,A,YR,F,HLY_0,CH,56.7,57.8,57.7,59.4,59.5,60.0,60.7,59.3,56.7,55.4
4,A,YR,F,HLY_0,CY,66.1,63.4,68.8,65.8,62.4,63.0,63.1,66.8,66.3,65.7


In [5]:

# convert from wide to long format

df_long = df_clean.melt(
    id_vars=meta_names, #remain fixed 
    var_name="year",
    value_name="value"
)

df_long.head(20)

,freq,unit,sex,indic_he,geo,year,value
0,A,YR,F,HLY_0,AT,2014,57.8
1,A,YR,F,HLY_0,BE,2014,63.7
2,A,YR,F,HLY_0,BG,2014,66.1
3,A,YR,F,HLY_0,CH,2014,56.7
4,A,YR,F,HLY_0,CY,2014,66.1
5,A,YR,F,HLY_0,CZ,2014,65.0
6,A,YR,F,HLY_0,DE,2014,56.5
7,A,YR,F,HLY_0,DK,2014,61.4
8,A,YR,F,HLY_0,EE,2014,57.1
9,A,YR,F,HLY_0,EL,2014,64.9


In [6]:
# We only want to keep columns where years match healthy life years at birth
df_hly = df_long[
    (df_long["unit"] == "YR") &
    (df_long["indic_he"] == "HLY_0")
].copy()

In [7]:
# Remove EU aggregates
df_hly = df_hly[~df_hly["geo"].str.startswith("EU")]

# Keep total sex 
df_total = df_hly[df_hly["sex"] == "T"].copy()

df_total.sort_values(["geo", "year"]).head()
df_total["year"].min(), df_total["year"].max()
df_total["geo"].nunique()

31

In [8]:
df_total.head(20)

,freq,unit,sex,indic_he,geo,year,value
198,A,YR,T,HLY_0,AT,2014,57.7
199,A,YR,T,HLY_0,BE,2014,64.1
200,A,YR,T,HLY_0,BG,2014,64.0
201,A,YR,T,HLY_0,CH,2014,58.6
202,A,YR,T,HLY_0,CY,2014,66.0
203,A,YR,T,HLY_0,CZ,2014,64.1
204,A,YR,T,HLY_0,DE,2014,56.5
205,A,YR,T,HLY_0,DK,2014,60.9
206,A,YR,T,HLY_0,EE,2014,55.2
207,A,YR,T,HLY_0,EL,2014,64.5


In [9]:
# We want to narrow our df down to healthy life years for total population

df_target = df_long[
    (df_long["sex"] == "T") & # keep total population 
    (df_long["indic_he"] == "HLY_0")
].copy()

In [63]:
# rename column
df_target = df_target.rename(columns={"value": "healthy_life_years"})

In [10]:
df_target.head(10)

,freq,unit,sex,indic_he,geo,year,value
198,A,YR,T,HLY_0,AT,2014,57.7
199,A,YR,T,HLY_0,BE,2014,64.1
200,A,YR,T,HLY_0,BG,2014,64.0
201,A,YR,T,HLY_0,CH,2014,58.6
202,A,YR,T,HLY_0,CY,2014,66.0
203,A,YR,T,HLY_0,CZ,2014,64.1
204,A,YR,T,HLY_0,DE,2014,56.5
205,A,YR,T,HLY_0,DK,2014,60.9
206,A,YR,T,HLY_0,EE,2014,55.2
207,A,YR,T,HLY_0,EL,2014,64.5


# Load feature data

In [12]:
# Load tsv.gz files
gdp_df = pd.read_csv("gdp.tsv.gz",sep="\t",compression="gzip")
hlt_exp_df = pd.read_csv("health_expen.tsv.gz",sep="\t",compression="gzip")

# CSV files
smoking = pd.read_csv("smoking.csv")
unmet_need_med_att = pd.read_csv("unmet_need_med_att.csv")
perc_hlt = pd.read_csv("perceived_health.csv")
anitbiotics = pd.read_csv("antibiotics.csv")
obesity = pd.read_csv("obesity_bmi.csv")
above_65 = pd.read_csv("above_65.csv")
life_expectancy = pd.read_csv("life_expec.csv")

pm = pd.read_csv("pm.csv")

gdp_df.head(10)


,"freq,unit,na_item,geo\TIME_PERIOD",2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,"A,CP_EUR_HAB,B1GQ,AL",3770,4100,4540,4880,4710,5420,6500,:,:,:
1,"A,CP_EUR_HAB,B1GQ,AT",40690,41760,43360,44570,42650,45380,49640,52330,53830,55710
2,"A,CP_EUR_HAB,B1GQ,BA",0,0,0,10,10,10,10,10,10,:
3,"A,CP_EUR_HAB,B1GQ,BE",37810,38980,40210,41730,40190,43680,48090,51090,52310 p,53930 p
4,"A,CP_EUR_HAB,B1GQ,BG",7070,7720,8340,9250,9440,10960,13310,14660,16260,18060 p
5,"A,CP_EUR_HAB,B1GQ,CH",75360,74010,73630,76680,76710,81610,92900,96280,99430,101810
6,"A,CP_EUR_HAB,B1GQ,CY",21930,23230,24660,26110,24630,27850,31560,33870,35670 p,36850 p
7,"A,CP_EUR_HAB,B1GQ,CZ",17040,18700,20270,21740,20980,23430,26670,29330,29440,31810
8,"A,CP_EUR_HAB,B1GQ,DE",39030,40630,41800,43030,42020,44910,48340 p,50660 p,51830 p,53520 p
9,"A,CP_EUR_HAB,B1GQ,DK",49270,51060,51950,53040,53540,58640,64430,62910,65650,68310


### Clean GDP data

In [13]:
gdp_df.columns[0]

'freq,unit,na_item,geo\\TIME_PERIOD'

In [14]:
# create metadate names
gdp_meta_names = (
    gdp_df.columns[0]
    .replace("\\TIME_PERIOD", "")
    .split(",")
)

gdp_meta_names

['freq', 'unit', 'na_item', 'geo']

In [15]:
# split the first column 
gdp_meta_cols = gdp_df.iloc[:, 0].str.split(",", expand=True)

gdp_meta_cols.columns = gdp_meta_names

# remove original first column
gdp_df_clean = gdp_df.drop(gdp_df.columns[0], axis=1)

#combine metadata and date
gdp_df_clean = pd.concat(
    [gdp_meta_cols, gdp_df_clean],
    axis=1
)

gdp_df_clean.head()

,freq,unit,na_item,geo,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,CP_EUR_HAB,B1GQ,AL,3770,4100,4540,4880,4710,5420,6500,:,:,:
1,A,CP_EUR_HAB,B1GQ,AT,40690,41760,43360,44570,42650,45380,49640,52330,53830,55710
2,A,CP_EUR_HAB,B1GQ,BA,0,0,0,10,10,10,10,10,10,:
3,A,CP_EUR_HAB,B1GQ,BE,37810,38980,40210,41730,40190,43680,48090,51090,52310 p,53930 p
4,A,CP_EUR_HAB,B1GQ,BG,7070,7720,8340,9250,9440,10960,13310,14660,16260,18060 p


In [16]:
# wide to long

gdp_long = gdp_df_clean.melt(
    id_vars=gdp_meta_names,
    var_name="year",
    value_name="gdp"
)

gdp_long.head()

,freq,unit,na_item,geo,year,gdp
0,A,CP_EUR_HAB,B1GQ,AL,2016,3770
1,A,CP_EUR_HAB,B1GQ,AT,2016,40690
2,A,CP_EUR_HAB,B1GQ,BA,2016,0
3,A,CP_EUR_HAB,B1GQ,BE,2016,37810
4,A,CP_EUR_HAB,B1GQ,BG,2016,7070


In [17]:
# clean year and GDP
gdp_long["year"] = gdp_long["year"].astype(int)

gdp_long["gdp"] = (
    gdp_long["gdp"]
    .astype(str)
    .str.replace(r"[^\d\.\-]", "", regex=True)
)

gdp_long["gdp"] = pd.to_numeric(gdp_long["gdp"], errors="coerce")

In [18]:
gdp_long.head()
gdp_long.info()
gdp_long["unit"].unique()
gdp_long["na_item"].unique()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10730 entries, 0 to 10729
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   freq     10730 non-null  object 
 1   unit     10730 non-null  object 
 2   na_item  10730 non-null  object 
 3   geo      10730 non-null  object 
 4   year     10730 non-null  int64  
 5   gdp      10213 non-null  float64
dtypes: float64(1), int64(1), object(4)
memory usage: 503.1+ KB


array(['B1GQ', 'P3', 'P31_S13', 'P31_S14', 'P3_S13'], dtype=object)

In [19]:
gdp_final = gdp_long[
    (gdp_long["na_item"] == "B1GQ") &
    (gdp_long["unit"] == "CP_PPS_EU27_2020_HAB")
].copy()

In [20]:
gdp_final.head()
#gdp_final.describe()

,freq,unit,na_item,geo,year,gdp
432,A,CP_PPS_EU27_2020_HAB,B1GQ,AL,2016,8480.7
433,A,CP_PPS_EU27_2020_HAB,B1GQ,AT,2016,36385.1
434,A,CP_PPS_EU27_2020_HAB,B1GQ,BA,2016,8.9
435,A,CP_PPS_EU27_2020_HAB,B1GQ,BE,2016,33631.0
436,A,CP_PPS_EU27_2020_HAB,B1GQ,BG,2016,14411.0


In [21]:
gdp_final = gdp_final[["geo", "year", "gdp"]]

### Clean Health Expenditure dataset

In [22]:
hlt_exp_df.columns[0]

'freq,unit,icha11_hc,geo\\TIME_PERIOD'

In [23]:
# Create metadata names
hlt_meta_names = (
    hlt_exp_df.columns[0]
    .replace("\\TIME_PERIOD", "")
    .split(",")
)

hlt_meta_names

['freq', 'unit', 'icha11_hc', 'geo']

In [24]:
# Split metadata columns
hlt_meta_cols = hlt_exp_df.iloc[:, 0].str.split(",", expand=True)

hlt_meta_cols.columns = hlt_meta_names

#remove original metadata column
hlt_exp_clean = hlt_exp_df.drop(
    hlt_exp_df.columns[0],
    axis=1
)

# combine back together
hlt_exp_clean = pd.concat(
    [hlt_meta_cols, hlt_exp_clean],
    axis=1
)

In [25]:
# melt to long value
hlt_exp_long = hlt_exp_clean.melt(
    id_vars=hlt_meta_names,
    var_name="year",
    value_name="health_exp"
)

In [26]:
# Clean year

hlt_exp_long["year"] = hlt_exp_long["year"].astype(int)

hlt_exp_long["health_exp"] = (
    hlt_exp_long["health_exp"]
    .astype(str)
    .str.replace(r"[^\d\.\-]", "", regex=True)
)

hlt_exp_long["health_exp"] = pd.to_numeric(
    hlt_exp_long["health_exp"],
    errors="coerce"
)

In [27]:
hlt_exp_long.head(20)

,freq,unit,icha11_hc,geo,year,health_exp
0,A,EUR_HAB,TOT_HC,AT,2013,3941.40
1,A,EUR_HAB,TOT_HC,BA,2013,337.05
2,A,EUR_HAB,TOT_HC,BE,2013,3761.74
3,A,EUR_HAB,TOT_HC,BG,2013,419.48
4,A,EUR_HAB,TOT_HC,CH,2013,6963.94
5,A,EUR_HAB,TOT_HC,CY,2013,1448.26
6,A,EUR_HAB,TOT_HC,CZ,2013,1136.77
7,A,EUR_HAB,TOT_HC,DE,2013,3819.65
8,A,EUR_HAB,TOT_HC,DK,2013,4759.80
9,A,EUR_HAB,TOT_HC,EA19,2013,NaN


In [28]:
hlt_exp_final = hlt_exp_long[
    (hlt_exp_long["unit"] == "EUR_HAB") &
    (hlt_exp_long["icha11_hc"] == "TOT_HC")
].copy()


In [29]:
hlt_exp_final.head()
hlt_exp_final.shape
hlt_exp_final.duplicated(subset=["geo", "year"]).sum()

np.int64(0)

In [30]:
# keep only column needed for merging

hlt_exp_final = hlt_exp_final[
    ["geo", "year", "health_exp"]
]

## Clean "Smoking"

In [31]:
# rename year column
smoking = smoking.rename(columns={"TIME_PERIOD":"year"})
smoking = smoking.rename(columns={"OBS_VALUE":"smokers"})

# only keep necessary info
smoking = smoking[
    ["geo", "year", "smokers"]]

In [32]:
# Linear interpolation for smoking to fill out empty years
# Make sure year is numeric
smoking["year"] = smoking["year"].astype(int)

# Keep only relevant years eventually, but first create full panel
geo = smoking["geo"].unique()
years = range(2014, 2024)

full_index = pd.MultiIndex.from_product(
    [geo, years],
    names=["geo", "year"]
)

# Add missing country-year rows
smoking_full = (
    smoking
    .set_index(["geo", "year"])
    .reindex(full_index)
    .reset_index()
)

# Sort before interpolation
smoking_full = smoking_full.sort_values(["geo", "year"])

#Interpolate within each country
smoking_full["smokers"] = (
    smoking_full
    .groupby("geo")["smokers"]
    .transform(lambda x: x.interpolate(method="linear"))
)

In [33]:
smoking_full.head(10)

,geo,year,smokers
0,AT,2014,26.000000
1,AT,2015,26.666667
2,AT,2016,27.333333
3,AT,2017,28.000000
4,AT,2018,27.000000
5,AT,2019,26.000000
6,AT,2020,25.000000
7,AT,2021,27.333333
8,AT,2022,29.666667
9,AT,2023,32.000000


## Clean "Unmet need for medical attention"

In [34]:
# rename year column
unmet_need_med_att = unmet_need_med_att.rename(columns={"TIME_PERIOD":"year"})
unmet_need_med_att = unmet_need_med_att.rename(columns={"OBS_VALUE":"med_att"})

# only keep necessary info
unmet_need_med_att = unmet_need_med_att[
    ["geo", "year", "med_att"]]


unmet_need_med_att.head(20)


,geo,year,med_att
0,AL,2017,13.1
1,AL,2018,14.8
2,AL,2019,14.6
3,AL,2020,10.6
4,AL,2021,10.7
5,AL,2022,10.2
6,AL,2023,10.3
7,AT,2008,0.7
8,AT,2009,0.5
9,AT,2010,0.6


## Clean Perceived Health data

In [35]:
perc_hlt = perc_hlt.rename(columns={"TIME_PERIOD":"year"})
perc_hlt = perc_hlt.rename(columns={"OBS_VALUE":"perc_health"})

# only keep necessary info
perc_hlt = perc_hlt[
    ["geo", "year", "perc_health"]]

perc_hlt.head(10)

,geo,year,perc_health
0,AL,2017,81.4
1,AL,2018,81.8
2,AL,2019,82.0
3,AL,2020,82.8
4,AL,2021,83.0
5,AL,2022,83.4
6,AL,2023,83.9
7,AT,2005,71.0
8,AT,2006,71.9
9,AT,2007,72.4


## Clean Antibiotics

In [36]:
anitbiotics = anitbiotics.rename(columns={"TIME_PERIOD":"year"})
anitbiotics = anitbiotics.rename(columns={"OBS_VALUE":"antibiotics_DD"})

# only keep necessary info
anitbiotics = anitbiotics[
    ["geo", "year", "antibiotics_DD"]]

anitbiotics.head(10)

,geo,year,antibiotics_DD
0,AT,2019,11.6
1,AT,2020,8.8
2,AT,2021,8.8
3,AT,2022,10.5
4,AT,2023,11.3
5,AT,2024,11.8
6,BE,2013,24.2
7,BE,2014,24.0
8,BE,2015,24.4
9,BE,2016,24.2


## Clean Obesity

In [37]:
obesity.head(10)

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,freq,Time frequency,unit,Unit of measure,bmi,Body Mass Index,geo,Geopolitical entity (reporting),TIME_PERIOD,Time,OBS_VALUE,Observation value,OBS_FLAG,Observation status (Flag) V2 structure,CONF_STATUS,Confidentiality status (flag)
0,dataflow,ESTAT:SDG_02_10(1.0),Obesity rate by body mass index (BMI),A,Annual,PC,Percentage,BMI_GE25,Overweight,AL,Albania,2022,NaN,57.4,NaN,NaN,NaN,NaN,NaN
1,dataflow,ESTAT:SDG_02_10(1.0),Obesity rate by body mass index (BMI),A,Annual,PC,Percentage,BMI_GE25,Overweight,AT,Austria,2008,NaN,49.3,NaN,NaN,NaN,NaN,NaN
2,dataflow,ESTAT:SDG_02_10(1.0),Obesity rate by body mass index (BMI),A,Annual,PC,Percentage,BMI_GE25,Overweight,AT,Austria,2014,NaN,48.0,NaN,NaN,NaN,NaN,NaN
3,dataflow,ESTAT:SDG_02_10(1.0),Obesity rate by body mass index (BMI),A,Annual,PC,Percentage,BMI_GE25,Overweight,AT,Austria,2017,NaN,50.0,NaN,NaN,NaN,NaN,NaN
4,dataflow,ESTAT:SDG_02_10(1.0),Obesity rate by body mass index (BMI),A,Annual,PC,Percentage,BMI_GE25,Overweight,AT,Austria,2019,NaN,52.2,NaN,NaN,NaN,NaN,NaN
5,dataflow,ESTAT:SDG_02_10(1.0),Obesity rate by body mass index (BMI),A,Annual,PC,Percentage,BMI_GE25,Overweight,AT,Austria,2022,NaN,52.7,NaN,NaN,NaN,NaN,NaN
6,dataflow,ESTAT:SDG_02_10(1.0),Obesity rate by body mass index (BMI),A,Annual,PC,Percentage,BMI_GE25,Overweight,AT,Austria,2025,NaN,52.9,NaN,NaN,NaN,NaN,NaN
7,dataflow,ESTAT:SDG_02_10(1.0),Obesity rate by body mass index (BMI),A,Annual,PC,Percentage,BMI_GE25,Overweight,BA,Bosnia and Herzegovina,2022,NaN,60.3,NaN,NaN,NaN,NaN,NaN
8,dataflow,ESTAT:SDG_02_10(1.0),Obesity rate by body mass index (BMI),A,Annual,PC,Percentage,BMI_GE25,Overweight,BE,Belgium,2008,NaN,47.5,NaN,NaN,NaN,NaN,NaN
9,dataflow,ESTAT:SDG_02_10(1.0),Obesity rate by body mass index (BMI),A,Annual,PC,Percentage,BMI_GE25,Overweight,BE,Belgium,2014,NaN,49.3,NaN,NaN,NaN,NaN,NaN


In [39]:
obesity = obesity.rename(columns={"TIME_PERIOD":"year"})
obesity = obesity.rename(columns={"OBS_VALUE":"overweight_pct"})

# only keep necessary info
obesity = obesity[
    ["geo", "year", "overweight_pct"]]

obesity.head(10)

,geo,year,overweight_pct
0,AL,2022,57.4
1,AT,2008,49.3
2,AT,2014,48.0
3,AT,2017,50.0
4,AT,2019,52.2
5,AT,2022,52.7
6,AT,2025,52.9
7,BA,2022,60.3
8,BE,2008,47.5
9,BE,2014,49.3


In [40]:
# Make sure year is numeric
obesity["year"] = obesity["year"].astype(int)

# Countries in the feature df
geo = obesity["geo"].unique()

# Years you want in final dataset
years = range(2016, 2024)  

full_index = pd.MultiIndex.from_product(
    [geo, years],
    names=["geo", "year"]
)

obesity_full = (
    obesity
    .set_index(["geo", "year"])
    .reindex(full_index)
    .reset_index()
)

obesity_full = obesity_full.sort_values(["geo", "year"])

obesity_full["overweight_pct"] = (
    obesity_full
    .groupby("geo")["overweight_pct"]
    .transform(lambda x: x.interpolate(method="linear"))
)

## Clean "Above 65"

In [41]:
above_65 = above_65.rename(columns={"TIME_PERIOD":"year"})
above_65 = above_65.rename(columns={"OBS_VALUE":"above_65"})

# only keep necessary info
above_65 = above_65[
    ["geo", "year", "above_65"]]

above_65.head(10)

,geo,year,above_65
0,AD,2019,13.6
1,AL,2014,12.0
2,AL,2015,12.4
3,AL,2016,12.8
4,AL,2017,13.1
5,AL,2018,13.6
6,AL,2019,14.1
7,AL,2020,14.8
8,AL,2021,15.2
9,AL,2022,15.7


### Clean "Life Expectancy at birth"

In [42]:
life_expectancy = life_expectancy.rename(columns={"TIME_PERIOD":"year"})
life_expectancy = life_expectancy.rename(columns={"OBS_VALUE":"life_expectancy"})

# only keep necessary info
life_expectancy = life_expectancy[
    ["geo", "year", "life_expectancy"]]

life_expectancy.head(10)

,geo,year,life_expectancy
0,AL,2013,77.9
1,AL,2014,78.2
2,AL,2015,77.9
3,AL,2016,78.5
4,AL,2017,78.5
5,AL,2018,78.9
6,AL,2019,79.1
7,AL,2020,77.4
8,AL,2021,75.5
9,AL,2022,79.1


## Clean PM2.5

In [45]:
pm = pm.rename(columns={"TIME_PERIOD":"year"})
pm = pm.rename(columns={"OBS_VALUE":"pm25_deaths"})

# only keep necessary info
pm = pm[
    ["geo", "year", "pm25_deaths"]]

pm.head(10)

,geo,year,pm25_deaths
0,AL,2005,6269
1,AL,2007,4844
2,AL,2008,4280
3,AL,2009,5099
4,AL,2010,6031
5,AL,2011,5572
6,AL,2012,4873
7,AL,2013,4233
8,AL,2014,3328
9,AL,2015,4692


## Merging the datasets

### Health expenditure with GDP

In [47]:
# Merge health expenditure df with GDP

# Restrict years
hlt_exp_final = hlt_exp_final[
    (hlt_exp_final["year"] >= 2016) &
    (hlt_exp_final["year"] <= 2023)
]

gdp_final = gdp_final[
    (gdp_final["year"] >= 2016) &
    (gdp_final["year"] <= 2023)
]

# Merge
merged_df = hlt_exp_final.merge(
    gdp_final,
    on=["geo", "year"],
    how="inner"
)

merged_df.head(20)


,geo,year,health_exp,gdp
0,AT,2016,4281.48,36385.1
1,BA,2016,402.60,8.9
2,BE,2016,4120.91,33631.0
3,BG,2016,527.51,14411.0
4,CY,2016,1459.24,24483.3
5,CZ,2016,1241.50,25443.3
6,DE,2016,4258.19,36023.5
7,DK,2016,5066.91,35992.7
8,EA19,2016,3263.22,30617.5
9,EA20,2016,3233.52,30466.1


### Smoking

In [48]:
countries = merged_df["geo"].unique()
years = range(2016, 2023)

full_index = pd.MultiIndex.from_product(
    [countries, years],
    names=["geo", "year"]
)

smoking_full = (
    smoking
    .set_index(["geo", "year"])
    .reindex(full_index)
    .reset_index()
    .sort_values(["geo", "year"])
)

smoking_full["smokers"] = (
    smoking_full
    .groupby("geo")["smokers"]
    .transform(lambda x: x.interpolate(method="linear").ffill().bfill())
)

In [49]:
merged_df = merged_df.merge(
    smoking_full[["geo", "year", "smokers"]],
    on=["geo", "year"],
    how="inner"
)

merged_df["year"].min(), merged_df["year"].max()

(np.int64(2016), np.int64(2022))

### Unmet need for medical attention

In [50]:
merged_df = merged_df.merge(
    unmet_need_med_att[["geo", "year", "med_att"]],
    on=["geo", "year"],
    how="left"
)

merged_df["year"].min(), merged_df["year"].max()

(np.int64(2016), np.int64(2022))

### Perceived health

In [51]:
merged_df = merged_df.merge(
    perc_hlt[["geo", "year", "perc_health"]],
    on=["geo", "year"],
    how="left"
)

merged_df["year"].min(), merged_df["year"].max()

(np.int64(2016), np.int64(2022))

### Antibiotics

In [52]:
merged_df = merged_df.merge(
    anitbiotics[["geo", "year", "antibiotics_DD"]],
    on=["geo", "year"],
    how="left"
)

merged_df["year"].min(), merged_df["year"].max()

(np.int64(2016), np.int64(2022))

### Obesity

In [53]:
countries = merged_df["geo"].unique()
years = range(2016, 2023)

full_index = pd.MultiIndex.from_product(
    [countries, years],
    names=["geo", "year"]
)

obesity_full = (
    obesity
    .set_index(["geo", "year"])
    .reindex(full_index)
    .reset_index()
    .sort_values(["geo", "year"])
)

obesity_full["overweight_pct"] = (
    obesity_full
    .groupby("geo")["overweight_pct"]
    .transform(lambda x: x.interpolate(method="linear").ffill().bfill())
)

In [54]:
merged_df = merged_df.merge(
    obesity_full[["geo", "year", "overweight_pct"]],
    on=["geo", "year"],
    how="left"
)

merged_df["year"].min(), merged_df["year"].max()

(np.int64(2016), np.int64(2022))

### Above 65

In [55]:
merged_df = merged_df.merge(
    above_65[["geo", "year", "above_65"]],
    on=["geo", "year"],
    how="left"
)

merged_df["year"].min(), merged_df["year"].max()

(np.int64(2016), np.int64(2022))

In [56]:
merged_df.head(10)

,geo,year,health_exp,gdp,smokers,med_att,perc_health,antibiotics_DD,overweight_pct,above_65
0,AT,2016,4281.48,36385.1,28.0,0.2,70.3,NaN,50.0,18.4
1,BA,2016,402.60,8.9,NaN,NaN,NaN,NaN,60.3,NaN
2,BE,2016,4120.91,33631.0,19.0,2.5,73.9,24.2,48.6,18.2
3,BG,2016,527.51,14411.0,36.0,2.8,65.8,19.2,59.2,21.1
4,CY,2016,1459.24,24483.3,28.0,0.6,78.7,28.4,52.7,15.0
5,CZ,2016,1241.50,25443.3,29.0,0.7,60.4,NaN,62.3,18.3
6,DE,2016,4258.19,36023.5,25.0,0.3,65.2,NaN,56.7,21.1
7,DK,2016,5066.91,35992.7,19.0,1.3,71.3,17.0,54.7,18.8
8,EA19,2016,3263.22,30617.5,NaN,2.3,68.4,NaN,NaN,NaN
9,EA20,2016,3233.52,30466.1,NaN,2.3,68.3,NaN,50.2,19.9


In [57]:
countries = merged_df["geo"].unique()

### Life Expectancy

In [58]:
merged_df = merged_df.merge(
    life_expectancy[["geo", "year", "life_expectancy"]],
    on=["geo", "year"],
    how="left"
)

merged_df["year"].min(), merged_df["year"].max()

(np.int64(2016), np.int64(2022))

## PM2.5

In [59]:
merged_df = merged_df.merge(
    pm[["geo", "year", "pm25_deaths"]],
    on=["geo", "year"],
    how="left"
)

merged_df["year"].min(), merged_df["year"].max()

(np.int64(2016), np.int64(2022))

### Merge with target df

In [64]:
# merge with target df
merged_df["year"] = merged_df["year"].astype(int)
df_target["year"] = df_target["year"].astype(int)

merged_df["geo"] = merged_df["geo"].astype(str).str.strip()
df_target["geo"] = df_target["geo"].astype(str).str.strip()

merged_df = merged_df.merge(
    df_target[["geo", "year", "healthy_life_years"]],
    on=["geo", "year"],
    how="inner"
)

merged_df["year"].min(), merged_df["year"].max()

(np.int64(2016), np.int64(2022))

In [65]:
# remove anything in the geo column that is not two characters
merged_df = merged_df[
    merged_df["geo"].astype(str).str.len() == 2
]

In [66]:
merged_df.head(20)

,geo,year,health_exp,gdp,smokers,med_att,perc_health,antibiotics_DD,overweight_pct,above_65,life_expectancy,pm25_deaths,healthy_life_years
0,AT,2016,4281.48,36385.1,28.0,0.2,70.3,NaN,50.0,18.4,81.8,3911.0,57.0
1,BE,2016,4120.91,33631.0,19.0,2.5,73.9,24.2,48.6,18.2,81.5,5737.0,63.7
2,BG,2016,527.51,14411.0,36.0,2.8,65.8,19.2,59.2,21.1,74.9,12330.0,65.7 b
3,CY,2016,1459.24,24483.3,28.0,0.6,78.7,28.4,52.7,15.0,82.7,534.0,68.2
4,CZ,2016,1241.50,25443.3,29.0,0.7,60.4,NaN,62.3,18.3,79.1,8433.0,63.3
5,DE,2016,4258.19,36023.5,25.0,0.3,65.2,NaN,56.7,21.1,81.0,43783.0,66.4
6,DK,2016,5066.91,35992.7,19.0,1.3,71.3,17.0,54.7,18.8,80.9,1618.0,60.3
7,EE,2016,1107.80,22190.6,23.0,15.3,52.9,12.0,56.1,19.0,NaN,128.0,56.8
8,EL,2016,1368.10,19106.0,37.0,13.1,74.0,33.1,54.8,21.1,81.5,11683.0,64.3
9,ES,2016,2178.67,26145.0,27.0,0.5,72.5,27.5,51.6,18.7,83.5,16920.0,66.2


# Clean up

In [67]:
# Remove the b's from the healthy life years column 

merged_df["healthy_life_years"] = (
    merged_df["healthy_life_years"]
    .astype(str)
    .str.extract(r'(\d+\.?\d*)')[0]
    .astype(float)
)

In [68]:
# Clean anything numeric

for col in merged_df.columns[2:]:
    merged_df[col] = pd.to_numeric(merged_df[col], errors="coerce")

In [69]:
# Check missingness
merged_df.isna().sum()

geo                    0
year                   0
health_exp             3
gdp                    3
smokers               14
med_att                7
perc_health            7
antibiotics_DD        21
overweight_pct         0
above_65               3
life_expectancy       11
pm25_deaths            7
healthy_life_years     9
dtype: int64

In [70]:
# N/A's - what we do is interpolate within countries when the gaps are small

cols_to_interp = [
    "health_exp",
    "gdp",
    "smokers",
    "med_att",
    "perc_health",
    "antibiotics_DD",
    "above_65",
    "life_expectancy"
]

merged_df = merged_df.sort_values(["geo", "year"])

for col in cols_to_interp:
    merged_df[col] = (
        merged_df.groupby("geo")[col]
        .transform(lambda x: x.interpolate(method="linear"))
    )

In [71]:
merged_df.isna().sum()

geo                    0
year                   0
health_exp             0
gdp                    0
smokers               14
med_att                0
perc_health            0
antibiotics_DD        21
overweight_pct         0
above_65               0
life_expectancy        7
pm25_deaths            7
healthy_life_years     9
dtype: int64

In [72]:
merged_df[merged_df["smokers"].isna()][["geo", "year", "smokers"]]

,geo,year,smokers
15,IS,2016,NaN
45,IS,2017,NaN
75,IS,2018,NaN
105,IS,2019,NaN
135,IS,2020,NaN
165,IS,2021,NaN
195,IS,2022,NaN
22,NO,2016,NaN
52,NO,2017,NaN
82,NO,2018,NaN


In [73]:
merged_df[merged_df["antibiotics_DD"].isna()][["geo", "year"]]

,geo,year
0,AT,2016
30,AT,2017
60,AT,2018
4,CZ,2016
34,CZ,2017
64,CZ,2018
5,DE,2016
35,DE,2017
65,DE,2018
95,DE,2019


In [74]:
merged_df.interpolate().ffill().bfill()

/var/folders/dv/yzy7j6xx27117cdh7g8rr8h80000gn/T/ipykernel_35284/1541848211.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  merged_df.interpolate().ffill().bfill()


,geo,year,health_exp,gdp,smokers,med_att,perc_health,antibiotics_DD,overweight_pct,above_65,life_expectancy,pm25_deaths,healthy_life_years
0,AT,2016,4281.48,36385.1,28.000000,0.2,70.3,11.6,50.000000,18.4,81.8,3911.0,57.0
30,AT,2017,4413.49,37012.2,28.000000,0.2,70.4,11.6,50.000000,18.5,81.7,4184.0,57.1
60,AT,2018,4558.54,38419.4,27.000000,0.1,71.7,11.6,51.100000,18.7,81.8,4981.0,56.9
90,AT,2019,4734.85,39263.8,26.000000,0.3,71.3,11.6,52.200000,18.8,82.0,3694.0,57.3
120,AT,2020,4867.79,37385.6,25.000000,0.1,74.0,8.8,52.366667,19.0,81.3,3152.0,58.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,UK,2018,3607.59,31789.2,15.333333,4.5,73.2,20.8,56.100000,18.2,81.3,3623.0,61.2
119,UK,2019,3838.53,32245.8,13.666667,4.5,73.2,20.8,56.100000,18.4,81.3,3623.0,61.2
149,UK,2020,3838.53,32245.8,12.000000,4.5,73.2,20.8,56.100000,18.4,81.3,3623.0,61.2
179,UK,2021,3838.53,32245.8,12.000000,4.5,73.2,20.8,56.100000,18.4,81.3,3623.0,61.2


In [75]:
merged_df.groupby("geo")["life_expectancy"].count()

geo
AT    7
BE    7
BG    7
CY    7
CZ    7
DE    7
DK    7
EE    0
EL    7
ES    7
FI    7
FR    7
HR    7
HU    7
IE    7
IS    7
IT    7
LT    7
LU    7
LV    7
MT    7
NL    7
NO    7
PL    7
PT    7
RO    7
SE    7
SI    7
SK    7
UK    7
Name: life_expectancy, dtype: int64

In [76]:
merged_df.groupby("geo")["antibiotics_DD"].count()

geo
AT    4
BE    7
BG    7
CY    7
CZ    4
DE    0
DK    7
EE    7
EL    7
ES    7
FI    7
FR    7
HR    7
HU    7
IE    7
IS    6
IT    7
LT    7
LU    7
LV    7
MT    7
NL    7
NO    7
PL    7
PT    7
RO    7
SE    7
SI    7
SK    7
UK    0
Name: antibiotics_DD, dtype: int64

# Create health gap column - how many years do people life "unhealthy"

In [77]:
# Life expectancy - healthy life years = health gap

merged_df["health_gap"] = (
    merged_df["life_expectancy"] -
    merged_df["healthy_life_years"]
)

In [79]:
# save as csv
merged_df.to_csv('df.csv', index=False)
merged_df.head(10)


,geo,year,health_exp,gdp,smokers,med_att,perc_health,antibiotics_DD,overweight_pct,above_65,life_expectancy,pm25_deaths,healthy_life_years,health_gap
0,AT,2016,4281.48,36385.1,28.000000,0.2,70.3,NaN,50.000000,18.4,81.8,3911.0,57.0,24.8
30,AT,2017,4413.49,37012.2,28.000000,0.2,70.4,NaN,50.000000,18.5,81.7,4184.0,57.1,24.6
60,AT,2018,4558.54,38419.4,27.000000,0.1,71.7,NaN,51.100000,18.7,81.8,4981.0,56.9,24.9
90,AT,2019,4734.85,39263.8,26.000000,0.3,71.3,11.6,52.200000,18.8,82.0,3694.0,57.3,24.7
120,AT,2020,4867.79,37385.6,25.000000,0.1,74.0,8.8,52.366667,19.0,81.3,3152.0,58.7,22.6
150,AT,2021,5527.17,40185.6,25.000000,0.3,72.3,8.8,52.533333,19.2,81.3,3185.0,61.8,19.5
180,AT,2022,5561.88,44569.5,25.000000,0.5,70.1,10.5,52.700000,19.4,81.4,3332.0,60.9,20.5
1,BE,2016,4120.91,33631.0,19.000000,2.5,73.9,24.2,48.600000,18.2,81.5,5737.0,63.7,17.8
31,BE,2017,4264.61,34519.6,19.000000,2.2,74.5,22.8,48.600000,18.5,81.6,5649.0,63.7,17.9
61,BE,2018,4420.70,35579.3,19.666667,1.8,74.9,22.3,49.400000,18.7,81.7,5887.0,63.4,18.3


In [80]:
# check number of countries
merged_df["geo"].nunique()

30

In [81]:
sorted(merged_df["geo"].unique())

['AT',
 'BE',
 'BG',
 'CY',
 'CZ',
 'DE',
 'DK',
 'EE',
 'EL',
 'ES',
 'FI',
 'FR',
 'HR',
 'HU',
 'IE',
 'IS',
 'IT',
 'LT',
 'LU',
 'LV',
 'MT',
 'NL',
 'NO',
 'PL',
 'PT',
 'RO',
 'SE',
 'SI',
 'SK',
 'UK']

In [82]:
merged_df.groupby("geo").size().sort_values()

geo
AT    7
SI    7
SE    7
RO    7
PT    7
PL    7
NO    7
NL    7
MT    7
LV    7
LU    7
LT    7
IT    7
IS    7
IE    7
HU    7
HR    7
FR    7
FI    7
ES    7
EL    7
EE    7
DK    7
DE    7
CZ    7
CY    7
BG    7
BE    7
SK    7
UK    7
dtype: int64

In [83]:
eu27 = {
    "AT","BE","BG","HR","CY","CZ","DK","EE","FI","FR",
    "DE","EL","HU","IE","IT","LV","LT","LU","MT","NL",
    "PL","PT","RO","SK","SI","ES","SE"
}

current = set(merged_df["geo"].unique())

print("Extra:", current - eu27)
print("Missing:", eu27 - current)

Extra: {'UK', 'NO', 'IS'}
Missing: set()


In [84]:
merged_df.isna().mean().mean() * 100

np.float64(2.5170068027210886)